# Code search in Python: semantic + exact, over one table

Developers search code two very different ways:

- **"Jump to the symbol I already know"** — show me every function named `train`. This wants an **exact** match, not a ranking.
- **"Find the function that does X"** — *train a classifier on a folder of images* — where the query and the code may share **no words at all**. This wants **semantic** (vector) search.
- **"Where do we call this API?"** — functions whose body mentions `subprocess` or `socket`. This wants **keyword** (BM25) search over the code text.

Most stacks reach for three tools here: a symbol index, a vector database, and a text search engine — kept in sync over three copies of the code. [Infino](https://pypi.org/project/infino/) does all three over **one table, one copy of the data**: exact lookup, vector k-NN, BM25, and a fused **hybrid** search, all in the same engine.

We index Python functions from CodeSearchNet and run each mode side by side.

## Setup

In [1]:
# %pip install infino pyarrow sentence-transformers datasets

In [2]:
import sys
from pathlib import Path

# Make the repo's shared helpers importable.
sys.path.insert(0, str(Path.cwd().parent))

import shutil

import infino
import pyarrow as pa

from _shared.embedding import DIM, METRIC, embed, embed_query, as_vector_column
from _shared.loaders import load_code_search
from _shared.sql import sql_lit, query

## 1. Load Python functions

Each function carries its name, full source, docstring, one-line summary, and the repo + URL it came from. The **docstring** is the natural-language description we embed; the **code** is what we keyword-search; the **name** is what we look up exactly.

In [3]:
funcs = load_code_search(n=800)
print(f"loaded {len(funcs)} functions")
f = funcs[0]
print("name:    ", f["func_name"])
print("repo:    ", f["repo"])
print("summary: ", f["summary"][:90])

loaded 800 functions
name:     train
repo:     ageitgey/face_recognition
summary:  Train a k - nearest neighbors classifier for face recognition.


## 2. Index code, docstrings, names, and vectors in one table

One table, three retrieval surfaces over the same rows:

- `func_name` and `code` — full-text indexed, so they support exact lookup, token match, and BM25 ranking.
- `repo`, `url`, `language` — scalar columns, SQL-filterable.
- `emb` — the docstring embedded into a vector, for natural-language search.

In [4]:
DB_DIR = "./code_data"
shutil.rmtree(DB_DIR, ignore_errors=True)

db = infino.connect(DB_DIR)
schema = pa.schema([
    pa.field("func_name", pa.large_utf8(), nullable=False),
    pa.field("code", pa.large_utf8(), nullable=False),
    pa.field("docstring", pa.large_utf8(), nullable=False),
    pa.field("summary", pa.large_utf8(), nullable=False),
    pa.field("repo", pa.large_utf8(), nullable=False),
    pa.field("url", pa.large_utf8(), nullable=False),
    pa.field("language", pa.large_utf8(), nullable=False),
    pa.field("emb", pa.list_(pa.float32(), DIM), nullable=False),
])
spec = (
    infino.IndexSpec()
    .fts("code")
    .fts("func_name")
    .vector("emb", DIM, n_cent=128, metric=METRIC)
)
table = db.create_table("functions", schema, spec)

# Embed the docstrings: natural-language search matches a query against these.
embeddings = embed([f["docstring"] for f in funcs])


def column(key):
    return pa.array([f[key] for f in funcs], type=pa.large_utf8())


table.append(pa.record_batch([
    column("func_name"), column("code"), column("docstring"), column("summary"),
    column("repo"), column("url"), column("language"),
    as_vector_column(embeddings),
], schema=schema))
print("indexed", db.query_sql("SELECT COUNT(*) AS n FROM functions").to_pydict()["n"][0], "functions")

indexed 800 functions


## 3. Exact symbol lookup

You know the name — you want *every* function called that, across every repo, not a fuzzy ranking. `exact_match` returns the rows whose column equals the value exactly, unranked. Here is a name that several projects define, and the (very different) jobs each one does.

In [5]:
# Pick the function name defined in the most distinct repos.
top = db.query_sql(
    "SELECT func_name, COUNT(DISTINCT repo) r FROM functions "
    "GROUP BY func_name ORDER BY r DESC, COUNT(*) DESC LIMIT 1"
).to_pydict()
name = top["func_name"][0]

hits = table.exact_match(
    "func_name", name, projection=["func_name", "repo", "summary"]
).to_pydict()
print(f"exact match on func_name = {name!r}  ->  {len(hits['func_name'])} definitions\n")
for repo, summary in zip(hits["repo"], hits["summary"]):
    print(f"  {repo:<28} {summary[:62]}")

exact match on func_name = 'train'  ->  3 definitions

  ageitgey/face_recognition    Train a k - nearest neighbors classifier for face recognition.
  dmlc/xgboost                 Train a booster with given parameters.
  tensorflow/tensor2tensor     Train the model using hparams.


## 4. Semantic search: find a function by what it does

Now the opposite problem — you don't know the name, you know the *intent*. We embed a natural-language description and let vector k-NN find the closest functions by meaning, even when the query shares no tokens with the code. (Smaller cosine distance = closer.)

In [6]:
nl_query = "train a machine-learning classifier on a folder of images"
qv = embed_query(nl_query)

hits = table.vector_search(
    "emb", qv, k=5, projection=["func_name", "repo", "summary", "score"]
).to_pydict()
print(f"query: {nl_query!r}\n")
for fn, repo, summary, score in zip(
    hits["func_name"], hits["repo"], hits["summary"], hits["score"]
):
    print(f"  {score:.3f}  {fn:<22} {repo}")
    print(f"         {summary[:72]}")

query: 'train a machine-learning classifier on a folder of images'

  0.485  train                  ageitgey/face_recognition
         Train a k - nearest neighbors classifier for face recognition.
  0.490  ItemList.split_by_folder fastai/fastai
         Split the data depending on the folder ( train or valid ) in which the f
  0.499  ItemList.label_from_folder fastai/fastai
         Give a label to each filename depending on its folder.
  0.513  _generator             tensorflow/tensor2tensor
         Base problem example generator for Allen Brain Atlas problems.
  0.563  imagenet50             slundberg/shap
         This dataset is used to create a set of 50 images representative of Imag


## 5. Keyword vs meaning: the same query, two ways

This is where the modes diverge. Take a plain-English intent — *remove duplicate elements* — and run it both ways over the same table:

- **`bm25_search`** ranks by literal tokens in the **code**. It latches onto words like *remove* and *elements* and surfaces functions that happen to contain them — often unrelated.
- **`vector_search`** ranks by **meaning**. It finds the functions that actually de-duplicate, even though their names and bodies never say *remove* or *duplicate*.

In [7]:
intent = "remove duplicate elements"

keyword = table.bm25_search(
    "code", intent, 5, projection=["func_name", "repo"]
).to_pydict()
meaning = table.vector_search(
    "emb", embed_query(intent), k=5, projection=["func_name", "repo"]
).to_pydict()

print(f"intent: {intent!r}\n")
print("keyword (BM25 over code) — literal token matches:")
for fn, repo in zip(keyword["func_name"], keyword["repo"]):
    print(f"  {fn:<28} {repo}")
print("\nmeaning (vector over docstrings) — what the function does:")
for fn, repo in zip(meaning["func_name"], meaning["repo"]):
    print(f"  {fn:<28} {repo}")

intent: 'remove duplicate elements'

keyword (BM25 over code) — literal token matches:
  all_files                    mozilla/DeepSpeech
  SequentialAug.dumps          apache/incubator-mxnet
  _remove_attributes           apache/incubator-mxnet
  symlink_remove               explosion/spaCy
  read_dfu_file                micropython/micropython

meaning (vector over docstrings) — what the function does:
  Index.get_duplicates         pandas-dev/pandas
  MultiCategoryProcessor.generate_classes fastai/fastai
  _remove_attributes           apache/incubator-mxnet
  Index._get_unique_index      pandas-dev/pandas
  TagView.delete               apache/incubator-superset


## 6. Hybrid search: keyword + meaning, fused in one SQL call

The strongest code search combines both signals: the literal token *and* the semantic intent. `hybrid_search` runs BM25 over the code and vector k-NN over the docstring embeddings, then fuses them with Reciprocal Rank Fusion (RRF) — one SQL call, one engine, no client-side merge. Because it's SQL, we join straight back to recover any columns we want.

In [8]:
hybrid_query = "load a json configuration file from disk"
qvec = ",".join(str(x) for x in embed_query(hybrid_query))

sql = (
    f"SELECT f.func_name, f.repo, s.score "
    f"FROM hybrid_search('functions', 'code', {sql_lit(hybrid_query)}, "
    f"                    'emb', '{qvec}', 8) s "
    f"JOIN functions f ON s._id = f._id "
    f"ORDER BY s.score DESC LIMIT 8"
)
rows = query(db, sql)
print(f"query: {hybrid_query!r}  (BM25 + vector, RRF-fused)\n")
for fn, repo, score in zip(rows.get("func_name", []), rows.get("repo", []), rows.get("score", [])):
    print(f"  {score:.4f}  {fn:<25} {repo}")

query: 'load a json configuration file from disk'  (BM25 + vector, RRF-fused)

  0.0313  load_model_from_path      explosion/spaCy
  0.0310  GPT2Config.from_json_file huggingface/pytorch-pretrained-BERT
  0.0164  GPT2PreTrainedModel.from_pretrained huggingface/pytorch-pretrained-BERT
  0.0164  read_squad_examples       huggingface/pytorch-pretrained-BERT
  0.0161  BertPreTrainedModel.from_pretrained huggingface/pytorch-pretrained-BERT
  0.0161  iob2json                  explosion/spaCy
  0.0159  Settings._settings_from_file nvbn/thefuck
  0.0156  XGBModel.load_model       dmlc/xgboost


## What just happened

Four kinds of code search ran over **one Infino table**, one copy of the data:

- **`exact_match`** — precise symbol lookup across every repo, no ranking.
- **`vector_search`** — natural-language → code by meaning.
- **`bm25_search`** — literal keyword matches in code; fast, but blind to intent.
- **`hybrid_search`** — BM25 + vector fused with RRF, in a single SQL call.

The keyword-vs-meaning section is the point: lexical and semantic search surface *different* functions for the same query, and `hybrid_search` gives you both at once — over one copy of the data.

No symbol index, no separate vector database, no text-search cluster to keep in sync — the same rows are exactly matchable, full-text searchable, vector searchable, and SQL-queryable at once.